<a href="https://colab.research.google.com/github/Page0526/survival-prediction/blob/main/TCGA_RNA_seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [2]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

page0526_tcga_5_subset_path = kagglehub.dataset_download('page0526/tcga-5-subset')

print('Data source import complete.')


100%|██████████| 149M/149M [00:09<00:00, 16.7MB/s]

Extracting files...


Data source import complete.


In [3]:
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 13.6 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=65109c8fce3b288077dbff645eb49c9faa392bba6c29964e7ff4fdc61b180881
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [4]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from lifelines.utils import concordance_index
from sklearn.preprocessing import StandardScaler

In [6]:
blca_df = pd.read_csv(f"{page0526_tcga_5_subset_path}/tcga_blca_all_clean.csv")

In [7]:
blca_df.head()

,case_id,slide_id,site,is_female,oncotree_code,age,survival_months,censorship,train,NDUFS5_cnv,...,ZWINT_rnaseq,ZXDA_rnaseq,ZXDB_rnaseq,ZXDC_rnaseq,ZYG11A_rnaseq,ZYG11B_rnaseq,ZYX_rnaseq,ZZEF1_rnaseq,ZZZ3_rnaseq,TPTEP1_rnaseq
0,TCGA-2F-A9KO,TCGA-2F-A9KO-01Z-00-DX1.195576CF-B739-4BD9-B15...,2F,0,BLCA,63,24.11,0,1.0,-1,...,-0.8388,4.1375,3.9664,1.8437,-0.3959,-0.2561,-0.2866,1.8770,-0.3179,-0.3633
1,TCGA-2F-A9KP,TCGA-2F-A9KP-01Z-00-DX1.3CDF534E-958F-4467-AA7...,2F,0,BLCA,66,11.96,0,1.0,2,...,-0.1083,0.3393,0.2769,1.7320,-0.0975,2.6955,-0.6741,1.0323,1.2766,-0.3982
2,TCGA-2F-A9KP,TCGA-2F-A9KP-01Z-00-DX2.718C82A3-252B-498E-BFB...,2F,0,BLCA,66,11.96,0,1.0,2,...,-0.1083,0.3393,0.2769,1.7320,-0.0975,2.6955,-0.6741,1.0323,1.2766,-0.3982
3,TCGA-2F-A9KQ,TCGA-2F-A9KQ-01Z-00-DX1.1C8CB2DD-5CC6-4E99-A0F...,2F,0,BLCA,69,94.81,1,1.0,0,...,-0.4155,1.6846,0.7711,-0.3061,-0.5016,2.8548,-0.6171,-0.8608,-0.0486,-0.3962
4,TCGA-2F-A9KR,TCGA-2F-A9KR-01Z-00-DX1.D6A4BD2D-18F3-4FA6-827...,2F,1,BLCA,59,104.57,0,1.0,0,...,-0.8143,0.8344,1.5075,3.6068,-0.5004,-0.0747,-0.2185,-0.4379,1.6913,1.7748


In [8]:
blca_df.describe()

,is_female,age,survival_months,censorship,train,NDUFS5_cnv,MACF1_cnv,RNA5SP44_cnv,KIAA0754_cnv,BMP8A_cnv,...,ZWINT_rnaseq,ZXDA_rnaseq,ZXDB_rnaseq,ZXDC_rnaseq,ZYG11A_rnaseq,ZYG11B_rnaseq,ZYX_rnaseq,ZZEF1_rnaseq,ZZZ3_rnaseq,TPTEP1_rnaseq
count,437.000000,437.000000,437.000000,437.000000,437.000000,437.000000,437.000000,437.000000,437.000000,437.000000,...,437.000000,437.000000,437.000000,437.00000,437.000000,437.000000,437.000000,437.000000,437.000000,437.000000
mean,0.251716,67.997712,27.415057,0.540046,0.887872,0.144165,0.167048,0.160183,0.162471,0.167048,...,0.071681,0.065703,0.119302,0.32713,0.184549,0.169060,-0.021081,-0.274432,0.259078,-0.081070
std,0.434496,10.644700,25.702726,0.498965,0.315886,0.723012,0.733856,0.727550,0.731758,0.733856,...,1.068212,0.999137,1.152723,1.26924,1.677983,1.115441,0.980554,1.126970,1.815057,0.860026
min,0.000000,34.000000,0.490000,0.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.782200,-1.770100,-1.813000,-2.54570,-0.550900,-2.035600,-1.646200,-2.198300,-2.392500,-0.415700
25%,0.000000,60.000000,11.070000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.759300,-0.611600,-0.616600,-0.49620,-0.496200,-0.649100,-0.685500,-1.015700,-0.643600,-0.395400
50%,0.000000,69.000000,18.560000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,-0.020800,-0.124100,-0.163300,0.20220,-0.370900,-0.003700,-0.286600,-0.392100,0.074800,-0.359000
75%,1.000000,76.000000,32.980000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.660100,0.667900,0.565700,0.99830,0.186600,0.784400,0.381000,0.192200,0.875500,-0.230500
max,1.000000,90.000000,163.170000,1.000000,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,...,4.953500,7.411700,7.936400,7.56020,20.004000,4.634000,4.367100,6.027000,22.265100,8.025000


In [ ]:
"""
Survival Neural Network (SNN) for TCGA RNA-seq Multi-Cancer Datasets
Supports: BLCA, BRCA, GBMLGG, LUAD, UCEC

Architecture: Original SNN (SELU + AlphaDropout blocks) from PORPOISE/MCAT.
Survival:     Discrete-time NLL loss (hazard per time bin).
Features:     Variance filter → univariate log-rank top-K → StandardScaler.
Training:     Full-batch (Cox/NLL requires full risk set).
"""

import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import OrderedDict

from lifelines.statistics import logrank_test
from lifelines.utils import concordance_index
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)


# ─────────────────────────────────────────────────────────────
# 1. CONFIG
# ─────────────────────────────────────────────────────────────

class Config:
    # ── Data ─────────────────────────────────────────────────
    DATA_DIR        = f"{page0526_tcga_5_subset_path}"
    # CANCER_TYPES    = ["BLCA", "BRCA", "GBMLGG", "LUAD", "UCEC"]
    CANCER_TYPE = "blca"
    CLINICAL_COLS   = ["is_female", "age"]
    SURVIVAL_COL    = "survival_months"
    EVENT_COL       = "censorship"      # 1 = event observed, 0 = censored
    TRAIN_COL       = "train"
    CNV_SUFFIX      = "_cnv"
    RNASEQ_SUFFIX   = "_rnaseq"

    # ── Feature selection ────────────────────────────────────
    VARIANCE_THRESHOLD  = 0.01
    TOP_K_CNV           = 100
    TOP_K_RNASEQ        = 400
    USE_PCA             = False
    PCA_COMPONENTS      = 128

    # ── SNN architecture (matches original paper) ────────────
    MODEL_SIZE      = "small"       # "small": [256,256,256,256]
                                    # "big":   [1024,1024,1024,256]
    N_CLASSES       = 4             # number of discrete survival time bins
    DROPOUT_FIRST   = 0.0           # first SNN_Block has no dropout (original)
    DROPOUT         = 0.25          # subsequent SNN_Block dropout

    # ── Training ─────────────────────────────────────────────
    EPOCHS          = 200
    LR              = 2e-4
    WEIGHT_DECAY    = 1e-5
    PATIENCE        = 30
    LR_PATIENCE     = 10
    LR_FACTOR       = 0.5
    LR_MIN          = 1e-6
    DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"

    # ── Misc ─────────────────────────────────────────────────
    SEED            = 42
    SAVE_DIR        = "./checkpoints"
    LOG_EVERY       = 10


# ─────────────────────────────────────────────────────────────
# 2. SNN BUILDING BLOCKS  (self-contained — no model_utils needed)
# ─────────────────────────────────────────────────────────────

class SNN_Block(nn.Module):
    """
    One block of the Self-Normalizing Neural Network.
    Linear → SELU → AlphaDropout
    SELU + AlphaDropout preserves the self-normalizing property
    (zero mean, unit variance activations through depth).
    """
    def __init__(self, dim1: int, dim2: int, dropout: float = 0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim1, dim2),
            nn.SELU(),
            nn.AlphaDropout(p=dropout)   # AlphaDropout, not regular Dropout
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


def init_max_weights(module: nn.Module):
    """
    Weight init from the original PORPOISE codebase.
    Uses a truncated normal scaled by the fan-in,
    which works well with SELU activations.
    """
    for m in module.modules():
        if isinstance(m, nn.Linear):
            stdv = 1. / np.sqrt(m.weight.size(1))
            nn.init.trunc_normal_(m.weight, std=stdv)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0.)


# ─────────────────────────────────────────────────────────────
# 3. SNN MODEL  (fixed + adapted from the original snippet)
# ─────────────────────────────────────────────────────────────

class SNN(nn.Module):
    """
    Genomic Fully-Connected SNN for discrete-time survival prediction.

    Input  : concatenated feature vector (clinical + CNV + RNA-seq)
    Output : (hazards, S, Y_hat)
               hazards : (1, n_classes)  sigmoid hazard per time bin
               S       : (1, n_classes)  cumulative survival S(t)
               Y_hat   : predicted time bin (argmax of hazard)

    Original bug fixed: super(MaxNet → SNN, self).__init__()
    """
    SIZE_DICT = {
        "small": [256, 256, 256, 256],
        "big":   [1024, 1024, 1024, 256],
    }

    def __init__(self, input_dim: int,
                 model_size: str = "small",
                 n_classes: int = 4,
                 dropout_first: float = 0.0,
                 dropout: float = 0.25):
        super(SNN, self).__init__()          # ← bug fix (was MaxNet)
        self.n_classes = n_classes
        hidden = self.SIZE_DICT[model_size]

        # First block has no dropout (input layer);
        # subsequent blocks use dropout=0.25 (matches original)
        fc_omic = [SNN_Block(dim1=input_dim,   dim2=hidden[0], dropout=dropout_first)]
        for i in range(len(hidden) - 1):
            fc_omic.append(SNN_Block(dim1=hidden[i], dim2=hidden[i + 1], dropout=dropout))

        self.fc_omic    = nn.Sequential(*fc_omic)
        self.classifier = nn.Linear(hidden[-1], n_classes)

        init_max_weights(self)

    def forward(self, x: torch.Tensor):
        """
        x : (B, input_dim)  — pre-scaled, concatenated feature tensor
        """
        features = self.fc_omic(x)                      # (B, hidden[-1])
        logits   = self.classifier(features)             # (B, n_classes)
        Y_hat    = torch.topk(logits, 1, dim=1)[1]      # (B, 1)
        hazards  = torch.sigmoid(logits)                 # (B, n_classes)
        S        = torch.cumprod(1 - hazards, dim=1)    # (B, n_classes)
        return hazards, S, Y_hat

    def relocate(self):
        """Move model to GPU(s). Call after construction."""
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if torch.cuda.device_count() > 1:
            device_ids = list(range(torch.cuda.device_count()))
            self.fc_omic = nn.DataParallel(
                self.fc_omic, device_ids=device_ids).to("cuda:0")
        else:
            self.fc_omic = self.fc_omic.to(device)
        self.classifier = self.classifier.to(device)


# ─────────────────────────────────────────────────────────────
# 4. DISCRETE-TIME SURVIVAL LOSS  (NLL)
# ─────────────────────────────────────────────────────────────

def nll_survival_loss(hazards: torch.Tensor,
                      S: torch.Tensor,
                      Y: torch.Tensor,
                      c: torch.Tensor,
                      alpha: float = 0.0) -> torch.Tensor:
    """
    Negative log-likelihood for discrete-time survival (Zadeh & Schmid 2020).

    hazards : (B, n_classes)  — sigmoid hazard per bin
    S       : (B, n_classes)  — cumulative survival = cumprod(1 - hazards)
    Y       : (B,)  long      — ground-truth time bin index
    c       : (B,)  float     — censoring: 1 = censored, 0 = event observed
                                (NOTE: opposite of EVENT_COL convention; flipped below)
    alpha   : weight for uncensored-only term (0 = pure NLL)

    For an observed event in bin t:
        L = -[ log h(t) + log S(t-1) ]   = -log[ h(t) * S(t-1) ]
    For a censored observation at bin t:
        L = -log S(t)
    """
    # Prepend S(0) = 1 so we can index S(t-1) as S_pad[:, t]
    S_pad    = torch.cat([torch.ones_like(S[:, :1]), S], dim=1)  # (B, n_classes+1)
    Y_idx    = Y.view(-1, 1)                                      # (B, 1)

    # S(t-1) and h(t) for each sample's event bin
    s_prev   = torch.gather(S_pad, 1, Y_idx).squeeze(1)          # (B,)
    h_t      = torch.gather(hazards, 1, Y_idx).squeeze(1)        # (B,)

    # c=1 → censored, c=0 → event (PORPOISE convention)
    # NLL for uncensored: -log(h(t)) - log(S(t-1))
    # NLL for censored:   -log(S(t))
    s_t      = torch.gather(S, 1, Y_idx).squeeze(1)              # (B,)

    l_uncens = -torch.log(h_t + 1e-7) - torch.log(s_prev + 1e-7)
    l_cens   = -torch.log(s_t + 1e-7)

    loss = (1.0 - c) * l_uncens + c * l_cens

    if alpha > 0:
        # Extra term: penalise uncertain predictions on uncensored cases
        loss = loss + alpha * (1.0 - c) * l_uncens

    return loss.mean()


def bin_survival_times(survival_times: np.ndarray,
                       n_bins: int,
                       eps: float = 1e-6,
                       cuts: Optional[np.ndarray] = None):
    """
    Assign each sample to a discrete time bin using quantile-based cuts.
    Cuts are computed on training data only, then reused for val/test.

    Returns (bin_indices, cuts)
    """
    if cuts is None:
        quantiles = np.linspace(0, 1, n_bins + 1)
        cuts = np.quantile(survival_times, quantiles)
        cuts[0]  -= eps
        cuts[-1] += eps
    indices = np.digitize(survival_times, bins=cuts[1:-1]).astype(int)
    indices = np.clip(indices, 0, n_bins - 1)
    return indices, cuts


# ─────────────────────────────────────────────────────────────
# 5. FEATURE SELECTION  (log-rank, ~100x faster than per-feature Cox)
# ─────────────────────────────────────────────────────────────

def logrank_selection(X_train: pd.DataFrame,
                      y_train: pd.DataFrame,
                      top_k: int,
                      survival_col: str,
                      event_col: str) -> list:
    """
    Rank features by log-rank p-value (median split).
    Much faster than univariate CoxPH while giving equivalent rankings.
    """
    print(f"    Log-rank (median split): {X_train.shape[1]} → top {top_k}",
          end=" … ", flush=True)
    T = y_train[survival_col].values
    E = y_train[event_col].values
    p_vals = {}

    for col in X_train.columns:
        vals   = X_train[col].values
        median = np.median(vals)
        lo, hi = vals <= median, vals > median
        if lo.sum() < 2 or hi.sum() < 2:
            p_vals[col] = 1.0
            continue
        try:
            p_vals[col] = logrank_test(
                T[lo], T[hi], E[lo], E[hi]).p_value
        except Exception:
            p_vals[col] = 1.0

    selected = sorted(p_vals, key=p_vals.get)[:top_k]
    print(f"done  (best p = {p_vals[selected[0]]:.2e})")
    return selected


def select_features(train_df: pd.DataFrame,
                    cnv_cols: list,
                    rnaseq_cols: list,
                    cfg: Config):
    y = train_df[[cfg.SURVIVAL_COL, cfg.EVENT_COL]]

    def var_filter(cols):
        vt = VarianceThreshold(threshold=cfg.VARIANCE_THRESHOLD)
        vt.fit(train_df[cols])
        return [c for c, ok in zip(cols, vt.get_support()) if ok]

    cnv_v = var_filter(cnv_cols)    if cnv_cols    else []
    rna_v = var_filter(rnaseq_cols) if rnaseq_cols else []
    print(f"  Variance filter : CNV {len(cnv_cols)}→{len(cnv_v)}, "
          f"RNA-seq {len(rnaseq_cols)}→{len(rna_v)}")

    if cfg.USE_PCA:
        all_g = cnv_v + rna_v
        pca   = PCA(n_components=min(cfg.PCA_COMPONENTS, len(all_g)),
                    random_state=cfg.SEED)
        pca.fit(train_df[all_g])
        ev = pca.explained_variance_ratio_.cumsum()[-1]
        print(f"  PCA : {len(all_g)} → {pca.n_components_} comps "
              f"({ev*100:.1f}% var)")
        return [], [], pca, all_g

    k_cnv = min(cfg.TOP_K_CNV,    len(cnv_v))
    k_rna = min(cfg.TOP_K_RNASEQ, len(rna_v))

    sel_cnv = logrank_selection(
        train_df[cnv_v], y, k_cnv,
        cfg.SURVIVAL_COL, cfg.EVENT_COL) if k_cnv > 0 else []
    sel_rna = logrank_selection(
        train_df[rna_v], y, k_rna,
        cfg.SURVIVAL_COL, cfg.EVENT_COL)

    return sel_cnv, sel_rna, None, None


# ─────────────────────────────────────────────────────────────
# 6. DATASET
# ─────────────────────────────────────────────────────────────

class SurvivalTensors:
    """Full-batch tensor container. No DataLoader needed for small n."""
    def __init__(self, df: pd.DataFrame,
                 feature_cols: list,
                 bin_indices: np.ndarray,
                 cfg: Config):
        dev = cfg.DEVICE
        self.X = torch.tensor(
            df[feature_cols].values, dtype=torch.float32).to(dev)
        # Y: discrete time bin
        self.Y = torch.tensor(bin_indices, dtype=torch.long).to(dev)
        # c: 1=censored, 0=event  (NLL convention; EVENT_COL is 1=event → flip)
        self.c = torch.tensor(
            1.0 - df[cfg.EVENT_COL].values, dtype=torch.float32).to(dev)
        # keep raw time & event for C-index
        self.T_raw = df[cfg.SURVIVAL_COL].values
        self.E_raw = df[cfg.EVENT_COL].values.astype(bool)

    def __len__(self):
        return len(self.Y)


# ─────────────────────────────────────────────────────────────
# 7. DATA LOADING & PRE-PROCESSING
# ─────────────────────────────────────────────────────────────

def load_and_preprocess(cfg: Config):
    frames = []
    # for ct in cfg.CANCER_TYPES:
    #     p = Path(cfg.DATA_DIR) / f"{ct}.csv"
    #     if p.exists():
    #         df = pd.read_csv(p); df["cancer_type"] = ct
    #         frames.append(df)
    #         print(f"  Loaded {ct}: {len(df):,} samples")

    # if not frames:
    #     mp = Path(cfg.DATA_DIR) / "merged.csv"
    #     if not mp.exists():
    #         raise FileNotFoundError(
    #             f"No data in '{cfg.DATA_DIR}'. "
    #             "Add per-cancer CSVs or merged.csv.")
    #     frames.append(pd.read_csv(mp))
    #     print(f"  Loaded merged.csv: {len(frames[0]):,} samples")
    ct = cfg.CANCER_TYPE
    p = Path(cfg.DATA_DIR) / f"tcga_{ct}_all_clean.csv"
    df = pd.read_csv(p)

    # df_all = pd.concat(frames, ignore_index=True)
    df_all = df
    df_all = df_all[pd.to_numeric(
        df_all[cfg.SURVIVAL_COL], errors="coerce").notna()].reset_index(drop=True)
    print(f"\nTotal samples      : {len(df_all):,}")

    cnv_cols    = [c for c in df_all.columns if c.endswith(cfg.CNV_SUFFIX)]
    rnaseq_cols = [c for c in df_all.columns if c.endswith(cfg.RNASEQ_SUFFIX)]
    all_feat    = cfg.CLINICAL_COLS + cnv_cols + rnaseq_cols

    for col in all_feat + [cfg.SURVIVAL_COL, cfg.EVENT_COL]:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")
    df_all = df_all.dropna(
        subset=all_feat + [cfg.SURVIVAL_COL, cfg.EVENT_COL])
    df_all = df_all[df_all[cfg.SURVIVAL_COL] > 0].reset_index(drop=True)
    print(f"After cleaning     : {len(df_all):,} samples")
    print(f"Raw features       : clinical={len(cfg.CLINICAL_COLS)}, "
          f"CNV={len(cnv_cols)}, RNA-seq={len(rnaseq_cols)}")

    # ── Train / val split ─────────────────────────────────────
    tc = pd.to_numeric(df_all[cfg.TRAIN_COL], errors="coerce")
    if tc.isna().all():
        mask = df_all[cfg.TRAIN_COL].astype(str).str.strip().str.lower() \
                     .isin(["1", "true", "yes"])
    else:
        mask = tc.fillna(0).astype(int) == 1

    train_df = df_all[mask].copy()
    val_df   = df_all[~mask].copy()
    print(f"Train / Val        : {len(train_df):,} / {len(val_df):,}")

    if len(train_df) == 0:
        raise ValueError(
            f"Train split empty. '{cfg.TRAIN_COL}' values: "
            f"{sorted(df_all[cfg.TRAIN_COL].dropna().unique())}")
    if len(val_df) == 0:
        from sklearn.model_selection import train_test_split
        idx_tr, idx_val = train_test_split(
            df_all.index, test_size=0.2, random_state=cfg.SEED)
        train_df = df_all.loc[idx_tr].copy()
        val_df   = df_all.loc[idx_val].copy()
        print(f"  val empty → 80/20 fallback: "
              f"{len(train_df):,} / {len(val_df):,}")

    # ── Feature selection (train only) ────────────────────────
    print("\nFeature selection …")
    sel_cnv, sel_rna, pca_obj, pca_src = select_features(
        train_df, cnv_cols, rnaseq_cols, cfg)

    feature_cols = cfg.CLINICAL_COLS + sel_cnv + sel_rna
    print(f"  Final features   : clinical={len(cfg.CLINICAL_COLS)}, "
          f"CNV={len(sel_cnv)}, RNA-seq={len(sel_rna)} "
          f"(total={len(feature_cols)})")

    # ── Scale ─────────────────────────────────────────────────
    scale_cols = pca_src if pca_obj else feature_cols
    scaler     = StandardScaler()
    train_df   = train_df.copy(); val_df = val_df.copy()
    train_df[scale_cols] = scaler.fit_transform(train_df[scale_cols])
    val_df[scale_cols]   = scaler.transform(val_df[scale_cols])

    if pca_obj:
        pca_cols = [f"pc_{i}" for i in range(pca_obj.n_components_)]
        for df in (train_df, val_df):
            comps = pca_obj.transform(df[pca_src])
            for j, pc in enumerate(pca_cols):
                df[pc] = comps[:, j]
        feature_cols = cfg.CLINICAL_COLS + pca_cols

    # ── Bin survival times (quantile bins on train) ───────────
    tr_bins, cuts = bin_survival_times(
        train_df[cfg.SURVIVAL_COL].values, cfg.N_CLASSES)
    va_bins, _    = bin_survival_times(
        val_df[cfg.SURVIVAL_COL].values, cfg.N_CLASSES, cuts=cuts)

    print(f"  Time bins (cuts) : "
          f"{np.round(cuts, 1).tolist()}")

    train_data = SurvivalTensors(train_df, feature_cols, tr_bins, cfg)
    val_data   = SurvivalTensors(val_df,   feature_cols, va_bins, cfg)

    meta = dict(feature_cols=feature_cols, scaler=scaler,
                pca_obj=pca_obj, pca_src=pca_src,
                sel_cnv=sel_cnv, sel_rna=sel_rna, cuts=cuts)
    return train_data, val_data, meta


# ─────────────────────────────────────────────────────────────
# 8. EVALUATION
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate(model: SNN, data: SurvivalTensors) -> dict:
    model.eval()
    hazards, S, Y_hat = model(data.X)

    # Risk score = mean hazard across bins (higher → worse survival)
    risk = hazards.mean(dim=1).cpu().numpy()

    ci = concordance_index(data.T_raw, -risk, data.E_raw)

    # Accuracy: predicted bin vs true bin
    acc = (Y_hat.squeeze(1).cpu() == data.Y.cpu()).float().mean().item()

    return {"cindex": ci, "bin_acc": acc}


# ─────────────────────────────────────────────────────────────
# 9. TRAINING
# ─────────────────────────────────────────────────────────────

def train(cfg: Config):
    torch.manual_seed(cfg.SEED)
    np.random.seed(cfg.SEED)
    os.makedirs(cfg.SAVE_DIR, exist_ok=True)

    # Override device in case Config was defined before GPU was active
    cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    print("=" * 62)
    print("Loading data …")
    train_data, val_data, meta = load_and_preprocess(cfg)

    input_dim = len(meta["feature_cols"])
    model = SNN(
        input_dim    = input_dim,
        model_size   = cfg.MODEL_SIZE,
        n_classes    = cfg.N_CLASSES,
        dropout_first= cfg.DROPOUT_FIRST,
        dropout      = cfg.DROPOUT,
    )
    model.relocate()    # handles DataParallel + moves to GPU

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nSNN  size={cfg.MODEL_SIZE}  "
          f"input={input_dim}  bins={cfg.N_CLASSES}")
    print(f"Device : {cfg.DEVICE}  |  Params : {n_params:,}")

    optimizer = optim.Adam(
        model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=cfg.LR_FACTOR,
        patience=cfg.LR_PATIENCE, min_lr=cfg.LR_MIN)

    best_ci      = 0.0
    patience_ctr = 0
    history      = {"train_loss": [], "train_ci": [],
                    "val_ci": [],    "val_acc": []}

    print("\n" + "=" * 62)
    print(f"Training (NLL discrete-time survival, full-batch) …\n")

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        optimizer.zero_grad()

        hazards, S, _ = model(train_data.X)
        loss = nll_survival_loss(
            hazards, S, train_data.Y, train_data.c)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        tr_metrics  = evaluate(model, train_data)
        val_metrics = evaluate(model, val_data)
        scheduler.step(val_metrics["cindex"])
        cur_lr = optimizer.param_groups[0]["lr"]

        history["train_loss"].append(loss.item())
        history["train_ci"].append(tr_metrics["cindex"])
        history["val_ci"].append(val_metrics["cindex"])
        history["val_acc"].append(val_metrics["bin_acc"])

        if epoch % cfg.LOG_EVERY == 0 or epoch == 1:
            print(f"Epoch {epoch:03d}/{cfg.EPOCHS}  "
                  f"loss={loss.item():.4f}  "
                  f"train_ci={tr_metrics['cindex']:.4f}  "
                  f"val_ci={val_metrics['cindex']:.4f}  "
                  f"val_acc={val_metrics['bin_acc']:.3f}  "
                  f"lr={cur_lr:.2e}")

        if val_metrics["cindex"] > best_ci:
            best_ci      = val_metrics["cindex"]
            patience_ctr = 0
            torch.save({
                "epoch":       epoch,
                "model_state": model.state_dict(),
                "val_cindex":  best_ci,
                "input_dim":   input_dim,
                "meta":        meta,
                "config":      cfg.__dict__,
            }, Path(cfg.SAVE_DIR) / "best_snn.pt")
        else:
            patience_ctr += 1
            if patience_ctr >= cfg.PATIENCE:
                print(f"\nEarly stopping at epoch {epoch} "
                      f"(patience {cfg.PATIENCE} exceeded)")
                break

    print(f"\n{'='*62}")
    print(f"Best Val C-index : {best_ci:.4f}")
    print(f"Checkpoint       : {cfg.SAVE_DIR}/best_snn.pt")
    return model, history


# ─────────────────────────────────────────────────────────────
# 10. INFERENCE
# ─────────────────────────────────────────────────────────────

def load_and_predict(checkpoint_path: str,
                     df_new: pd.DataFrame) -> np.ndarray:
    """Return per-sample risk scores. Higher = higher risk."""
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    cfg  = Config(); cfg.__dict__.update(ckpt["config"])
    meta = ckpt["meta"]

    scaler       = meta["scaler"]
    pca_obj      = meta.get("pca_obj")
    pca_src      = meta.get("pca_src")
    feature_cols = meta["feature_cols"]

    scale_cols = pca_src if pca_obj else feature_cols
    df_new = df_new.copy()
    df_new[scale_cols] = scaler.transform(df_new[scale_cols])

    if pca_obj:
        pca_cols = [f"pc_{i}" for i in range(pca_obj.n_components_)]
        comps = pca_obj.transform(df_new[pca_src])
        for j, pc in enumerate(pca_cols):
            df_new[pc] = comps[:, j]
        feature_cols = cfg.CLINICAL_COLS + pca_cols

    X = torch.tensor(df_new[feature_cols].values, dtype=torch.float32)
    model = SNN(ckpt["input_dim"], cfg.MODEL_SIZE,
                cfg.N_CLASSES, cfg.DROPOUT_FIRST, cfg.DROPOUT)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    with torch.no_grad():
        hazards, S, _ = model(X)
    return hazards.mean(dim=1).numpy()  # mean hazard as risk score


# ─────────────────────────────────────────────────────────────
# 11. PLOTTING
# ─────────────────────────────────────────────────────────────

def plot_history(history: dict, save_path: str = "./training_curves.png"):
    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))

        axes[0].plot(history["train_loss"], color="#e84393", lw=2)
        axes[0].set_title("NLL Loss (train)"); axes[0].set_xlabel("Epoch")
        axes[0].grid(alpha=0.3)

        axes[1].plot(history["train_ci"], color="#ff8c00",
                     lw=1.5, ls="--", label="train")
        axes[1].plot(history["val_ci"],   color="#1f77b4",
                     lw=2, label="val")
        axes[1].axhline(0.5, ls=":", color="gray", lw=1, label="random")
        axes[1].set_title("C-index"); axes[1].set_xlabel("Epoch")
        axes[1].set_ylim(0.4, 1.0); axes[1].legend(); axes[1].grid(alpha=0.3)

        axes[2].plot(history["val_acc"], color="#2ca02c", lw=2)
        axes[2].set_title("Val Bin Accuracy"); axes[2].set_xlabel("Epoch")
        axes[2].set_ylim(0, 1); axes[2].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        print(f"Curves saved → {save_path}")
    except ImportError:
        print("matplotlib not installed; skipping plot.")


# ─────────────────────────────────────────────────────────────
# 12. ENTRY POINT
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    cfg = Config()
    model, history = train(cfg)
    plot_history(history)

Loading data …

Total samples      : 437
After cleaning     : 437 samples
Raw features       : clinical=2, CNV=1656, RNA-seq=18369
Train / Val        : 388 / 49

Feature selection …
  Variance filter : CNV 1656→1656, RNA-seq 18369→18341
    Log-rank (median split): 1656 → top 100 … 